# 🎯 Aula 12 — Classificação e Aprendizagem Supervisionada

## Utilizando dados conhecidos para prever categorias

**Disciplina:** ISW-039 — Mineração de Dados  
**Curso:** Desenvolvimento de Software Multiplataforma (DSM)  
**Ambiente:** Google Colab  
**Linguagem:** Python  
**Bibliotecas:** Pandas, NumPy, Matplotlib e Scikit-learn

---

## 🎯 Objetivos da aula

Ao final desta aula, você deverá ser capaz de:

- Compreender o conceito de aprendizagem supervisionada;
- Diferenciar classificação de agrupamento;
- Identificar variável alvo (`target`) e variáveis preditoras (`features`);
- Preparar dados para um problema de classificação;
- Separar dados em treinamento e teste;
- Treinar um classificador;
- Utilizar o algoritmo **K-Nearest Neighbors (KNN)**;
- Fazer previsões;
- Interpretar a matriz de confusão;
- Compreender acurácia, precisão, recall e F1-score;
- Identificar problemas de classificação no próprio projeto.

> **Projeto didático:** continuaremos utilizando o monitoramento de motores elétricos. Agora teremos uma informação que não existia no clustering: um histórico indicando se uma leitura representava **falha** ou **operação normal**.


# 🧠 1. O que é aprendizagem supervisionada?

Na aprendizagem supervisionada, o algoritmo recebe exemplos em que já conhecemos a resposta.

Exemplo:

```text
Temperatura   Vibração   Corrente
    62           1.5       11.2       → Normal
    79           3.4       14.8       → Falha
    65           1.8       12.0       → Normal
```

O algoritmo aprende uma relação entre:

```text
ENTRADAS (X)
     ↓
  MODELO
     ↓
SAÍDA (y)
```

Depois podemos fornecer novos dados:

```text
Temperatura = 81
Vibração    = 3.5
Corrente    = 15.1
```

e perguntar:

> **Esse motor está normal ou apresenta risco de falha?**


# 🔵 2. Classificação × Clustering

Na aula anterior trabalhamos com **clustering**.

### Clustering

Não temos rótulos conhecidos:

```text
Dados → algoritmo → grupos
```

### Classificação

Temos exemplos rotulados:

```text
Dados + resposta conhecida
          ↓
        modelo
          ↓
     nova previsão
```

Exemplo:

```text
Clustering:
"Existem 3 grupos de motores."

Classificação:
"Este motor pertence à classe Falha."
```

Essa diferença é fundamental.


# 🏭 3. Problema da indústria

Nossa indústria possui históricos de manutenção.

Para cada leitura, sabemos se posteriormente foi registrada uma falha.

Temos:

- temperatura;
- vibração;
- corrente;
- tensão;
- RPM;
- falha.

A variável:

```text
falha
```

será nosso **target**.

As demais serão utilizadas como **features**.


# 💻 4. Preparando o ambiente

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

np.random.seed(42)

print("Ambiente preparado!")

# 📥 5. Criando uma base de motores

Vamos criar uma base simulada.

A variável `falha` será construída a partir de uma combinação de temperatura, vibração e corrente, representando um cenário didático de manutenção preditiva.


In [ ]:
n = 800

df = pd.DataFrame({
    "temperatura": np.random.normal(65, 8, n),
    "vibracao": np.random.normal(2.2, 0.7, n),
    "corrente": np.random.normal(13, 2, n),
    "tensao": np.random.normal(380, 4, n),
    "rpm": np.random.normal(1740, 15, n)
})

# Cenário didático de falha
risco = (
    (df["temperatura"] > 76) &
    (df["vibracao"] > 2.8)
) | (
    (df["corrente"] > 16) &
    (df["temperatura"] > 70)
)

df["falha"] = risco.astype(int)

df.head()

Vamos verificar a distribuição da variável alvo.


In [ ]:
df["falha"].value_counts()

In [ ]:
df["falha"].value_counts(normalize=True).mul(100).round(2)

Vamos visualizar as classes.


In [ ]:
contagem = df["falha"].value_counts().sort_index()

plt.figure(figsize=(7, 4))
plt.bar(["Normal", "Falha"], contagem.values)
plt.title("Distribuição das Classes")
plt.xlabel("Classe")
plt.ylabel("Quantidade")
plt.show()

# 🎯 6. Identificando X e y

Em um problema de classificação, normalmente temos:

```text
X → características utilizadas pelo modelo
y → resposta que queremos prever
```

Neste problema:

```text
X =
temperatura
vibracao
corrente
tensao
rpm

y =
falha
```


In [ ]:
X = df.drop("falha", axis=1)
y = df["falha"]

print("X:")
print(X.head())

print("\ny:")
print(y.head())

# ✂️ 7. Dividindo treino e teste

Vamos utilizar:

```text
80% → treinamento
20% → teste
```

Também vamos utilizar `stratify=y` para preservar a proporção das classes.


In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Treinamento:", X_treino.shape)
print("Teste:", X_teste.shape)

Verificando a distribuição:


In [ ]:
print("Treino:")
print(y_treino.value_counts(normalize=True).round(3))

print("\nTeste:")
print(y_teste.value_counts(normalize=True).round(3))

# 📏 8. Normalizando os dados

Vamos utilizar `StandardScaler`.

> O scaler deve ser ajustado (`fit`) somente com os dados de treinamento.

Depois utilizamos esse mesmo scaler para transformar treino e teste.


In [ ]:
scaler = StandardScaler()

X_treino_scaled = scaler.fit_transform(X_treino)
X_teste_scaled = scaler.transform(X_teste)

print("Dados normalizados.")

# 🤖 9. Primeiro modelo — KNN

Vamos utilizar o algoritmo:

> **K-Nearest Neighbors (KNN)**

A ideia é simples:

Para classificar um novo registro, o algoritmo procura os **K vizinhos mais próximos** e verifica a classe predominante entre eles.

Exemplo:

```text
Novo ponto
    ↓
encontra vizinhos
    ↓
3 Normal
2 Falha
    ↓
Previsão = Normal
```

O KNN é interessante para a aula porque permite visualizar intuitivamente a relação entre distância e classificação.


# 🔧 10. Treinando o KNN

Vamos começar com:

```text
K = 5
```


In [ ]:
modelo_knn = KNeighborsClassifier(n_neighbors=5)

modelo_knn.fit(
    X_treino_scaled,
    y_treino
)

print("Modelo treinado!")

# 🔮 11. Fazendo previsões



In [ ]:
y_pred = modelo_knn.predict(X_teste_scaled)

y_pred[:20]

Vamos comparar algumas previsões com os valores reais.


In [ ]:
resultado = pd.DataFrame({
    "real": y_teste.values,
    "previsto": y_pred
})

resultado.head(20)

# 📊 12. Acurácia

A primeira métrica que vamos conhecer é a **acurácia**.

```text
Acurácia =
previsões corretas
------------------
total de previsões
```

Vamos calcular.


In [ ]:
acuracia = accuracy_score(y_teste, y_pred)

print(f"Acurácia: {acuracia:.2%}")

A acurácia é útil, mas não deve ser utilizada sozinha, principalmente quando as classes estão desbalanceadas.

Precisamos entender **quais erros o modelo está cometendo**.


# 🔲 13. Matriz de confusão

A matriz de confusão organiza as previsões:

| | Previsto Normal | Previsto Falha |
|---|---:|---:|
| **Real Normal** | Verdadeiro Negativo | Falso Positivo |
| **Real Falha** | Falso Negativo | Verdadeiro Positivo |

No problema industrial:

### Falso Positivo

O modelo diz:

> "Falha"

mas o motor estava normal.

### Falso Negativo

O modelo diz:

> "Normal"

mas o motor realmente apresentou falha.

Em manutenção preditiva, o falso negativo pode ser especialmente preocupante.


In [ ]:
cm = confusion_matrix(y_teste, y_pred)

print(cm)

Vamos visualizar a matriz.


In [ ]:
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Normal", "Falha"]
)

disp.plot()
plt.title("Matriz de Confusão — KNN")
plt.show()

# 📐 14. Precisão, Recall e F1-score

Vamos utilizar o relatório de classificação.


In [ ]:
print(
    classification_report(
        y_teste,
        y_pred,
        target_names=["Normal", "Falha"],
        zero_division=0
    )
)

### Precisão

Entre as previsões de determinada classe, quantas realmente pertenciam a ela?

### Recall

Entre os casos que realmente pertenciam à classe, quantos o modelo conseguiu encontrar?

### F1-score

É uma combinação entre precisão e recall.

No nosso problema:

> O **recall da classe Falha** merece atenção especial, pois queremos evitar deixar falhas reais passarem despercebidas.


# 🔍 15. Investigando o recall da classe Falha

Vamos calcular explicitamente.


In [ ]:
relatorio = classification_report(
    y_teste,
    y_pred,
    output_dict=True,
    zero_division=0
)

print("Recall da classe Falha:",
      round(relatorio["1"]["recall"], 3))

print("Precisão da classe Falha:",
      round(relatorio["1"]["precision"], 3))

print("F1 da classe Falha:",
      round(relatorio["1"]["f1-score"], 3))

# 🔧 16. Testando diferentes valores de K

O KNN depende do número de vizinhos.

Vamos testar vários valores:

```text
K = 1
K = 3
K = 5
K = 7
K = 9
K = 11
```


In [ ]:
resultados = []

for k in [1, 3, 5, 7, 9, 11]:
    modelo = KNeighborsClassifier(n_neighbors=k)
    modelo.fit(X_treino_scaled, y_treino)

    previsoes = modelo.predict(X_teste_scaled)

    resultados.append({
        "K": k,
        "Acurácia": accuracy_score(y_teste, previsoes)
    })

resultados_df = pd.DataFrame(resultados)

resultados_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    resultados_df["K"],
    resultados_df["Acurácia"],
    marker="o"
)
plt.title("Acurácia × Número de Vizinhos")
plt.xlabel("K")
plt.ylabel("Acurácia")
plt.xticks(resultados_df["K"])
plt.show()

O objetivo não é simplesmente escolher o K com maior acurácia.

Devemos considerar também:

- comportamento das classes;
- recall;
- precisão;
- F1-score;
- custo dos erros;
- objetivo do projeto.


# 🧪 17. Criando uma nova leitura

Imagine que chegou uma nova leitura de sensor:

```text
Temperatura = 82 °C
Vibração    = 3.4
Corrente    = 15.5 A
Tensão      = 380 V
RPM         = 1735
```

Vamos pedir ao modelo uma previsão.


In [ ]:
novo_motor = pd.DataFrame({
    "temperatura": [82],
    "vibracao": [3.4],
    "corrente": [15.5],
    "tensao": [380],
    "rpm": [1735]
})

novo_motor_scaled = scaler.transform(novo_motor)

previsao = modelo_knn.predict(novo_motor_scaled)

print("Classe prevista:", previsao[0])

if previsao[0] == 1:
    print("Resultado: possível FALHA")
else:
    print("Resultado: NORMAL")

### ⚠️ Atenção

Essa previsão é apenas didática.

Em um sistema industrial real, seria necessário:

- validar o modelo;
- utilizar dados reais;
- conhecer as especificações do equipamento;
- avaliar custos de falsos positivos e negativos;
- monitorar o modelo ao longo do tempo.


# 🧠 18. Classificação não significa necessariamente "falha"

O mesmo conceito pode ser aplicado a muitos problemas.

Exemplos:

```text
Fraude / Não fraude

Spam / Não spam

Cliente ativo / Inativo

Aprovado / Reprovado

Risco alto / Baixo

Defeito / Sem defeito

Doente / Não doente
```

O algoritmo muda conforme o problema, mas a lógica continua:

```text
DADOS ROTULADOS
      ↓
TREINAMENTO
      ↓
MODELO
      ↓
NOVOS DADOS
      ↓
PREVISÃO
```


# ⚠️ 19. Classificação e desbalanceamento

Na Aula 9 estudamos balanceamento.

Agora podemos perceber por que ele é importante.

Imagine:

```text
9.900 → Normal
  100 → Falha
```

Um modelo que sempre responde "Normal" teria:

```text
99% de acurácia
```

mas não encontraria nenhuma falha.

Por isso, para problemas desbalanceados devemos analisar:

- matriz de confusão;
- precisão;
- recall;
- F1-score;
- métricas específicas da classe minoritária.

A métrica escolhida deve refletir o problema de negócio.


# 📝 20. Exercícios

## Exercício 1 — Conceitos

Explique a diferença entre:

**A)** aprendizagem supervisionada;

**B)** aprendizagem não supervisionada.


In [ ]:
# Sua resposta



## Exercício 2 — X e y

Identifique:

- quais são as features;
- qual é o target;

no problema de previsão de falhas.


In [ ]:
# Sua resposta



## Exercício 3 — Treino e teste

Faça uma divisão:

```text
75% → treinamento
25% → teste
```

Utilize `stratify`.


In [ ]:
# Sua resposta



## Exercício 4 — KNN

Treine um KNN utilizando:

```text
K = 3
```

Calcule a acurácia.


In [ ]:
# Sua resposta



## Exercício 5 — Matriz de confusão

Gere a matriz de confusão do modelo do exercício anterior.

Quantos:

- verdadeiros positivos;
- verdadeiros negativos;
- falsos positivos;
- falsos negativos

foram encontrados?


In [ ]:
# Sua resposta



## Exercício 6 — Métricas

Calcule:

- precisão da classe Falha;
- recall da classe Falha;
- F1-score da classe Falha.


In [ ]:
# Sua resposta



## Exercício 7 — Diferentes K

Teste:

```text
K = 1
K = 5
K = 10
K = 20
```

Compare os resultados.


In [ ]:
# Sua resposta



## Exercício 8 — Nova previsão

Crie uma nova leitura de motor com seus próprios valores.

Utilize o modelo para prever:

```text
Normal
ou
Falha
```


In [ ]:
# Sua resposta



## Exercício 9 — Análise do erro

Imagine que um modelo apresenta:

```text
Acurácia = 96%
Recall da Falha = 62%
```

Esse modelo seria necessariamente bom para manutenção preditiva?

Justifique.


In [ ]:
# Sua resposta



## Exercício 10 — Decisão de negócio

Para uma empresa, o custo de deixar uma falha passar é muito maior do que o custo de enviar um técnico para verificar um motor que está normal.

Qual métrica deveria receber atenção especial?

Explique.


In [ ]:
# Sua resposta



# 🚀 21. Desafio — Primeiro classificador do seu projeto

Agora aplique classificação ao **seu projeto individual**, caso o problema possua uma variável alvo categórica.

### Etapa 1 — Defina o problema

Complete:

```text
Quero prever:
____________________________
```

### Etapa 2 — Defina o target

```text
Target:
____________________________
```

### Etapa 3 — Escolha as features

Escolha pelo menos **3 variáveis**.

### Etapa 4 — Separe os dados

```text
Treino
Teste
```

### Etapa 5 — Treine um KNN

Teste pelo menos:

```text
K = 3
K = 5
K = 7
K = 9
```

### Etapa 6 — Avalie

Apresente:

- acurácia;
- matriz de confusão;
- precisão;
- recall;
- F1-score.

### Etapa 7 — Interprete

Explique:

> **O modelo é adequado para o problema? Por quê?**


In [ ]:
# Desenvolva o classificador do seu projeto aqui.



# 🏭 22. Aplicação no projeto didático

No exemplo dos motores, nosso fluxo ficou:

```text
Sensores
   ↓
Dados históricos
   ↓
Limpeza / preparação
   ↓
Features
   ↓
Target = Falha
   ↓
Treinamento
   ↓
KNN
   ↓
Previsão
   ↓
Matriz de Confusão
   ↓
Avaliação
```

Esse fluxo será cada vez mais próximo de um projeto real de Mineração de Dados.

No projeto final do aluno, a estrutura poderá ser diferente, mas a lógica deverá permanecer:

```text
PROBLEMA
   ↓
DADOS
   ↓
PREPARAÇÃO
   ↓
MODELO
   ↓
AVALIAÇÃO
   ↓
CONCLUSÃO
```


# 📌 23. Checklist da Aula

- [ ] Entendo aprendizagem supervisionada;
- [ ] Sei diferenciar classificação de clustering;
- [ ] Sei identificar features e target;
- [ ] Sei separar treino e teste;
- [ ] Sei utilizar `StandardScaler`;
- [ ] Sei treinar um KNN;
- [ ] Sei realizar previsões;
- [ ] Sei calcular acurácia;
- [ ] Sei interpretar uma matriz de confusão;
- [ ] Entendo precisão;
- [ ] Entendo recall;
- [ ] Entendo F1-score;
- [ ] Sei testar diferentes valores de K;
- [ ] Entendo a importância dos falsos negativos;
- [ ] Consigo aplicar classificação ao meu projeto.

---

# 🎯 Conclusão

A sequência da disciplina está agora:

```text
Aula 3  → Pandas
Aula 4  → Limpeza
Aula 5  → ETL
Aula 6  → Web Scraping
Aula 7  → Banco de Dados + SQL
Aula 8  → Análise Exploratória
Aula 9  → Amostragem + Balanceamento
Aula 10 → Visualização
Aula 11 → Clustering
Aula 12 → Classificação
```

Já trabalhamos com:

```text
APRENDIZAGEM NÃO SUPERVISIONADA
            ↓
        CLUSTERING

APRENDIZAGEM SUPERVISIONADA
            ↓
       CLASSIFICAÇÃO
```

Na próxima aula vamos aprofundar os modelos de classificação e comparar diferentes algoritmos para o mesmo problema.

> 🤖 **Próxima aula: Árvores de Decisão e comparação de modelos.**
